In [2]:
print("Installing Apache Airflow... (This may take around 30-45 seconds)")
!pip install apache-airflow > /dev/null 2>&1
print("✔ Apache Airflow installed successfully!\n")

Installing Apache Airflow... (This may take around 30-45 seconds)
✔ Apache Airflow installed successfully!



In [13]:
import os
from datetime import datetime
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator

def create_transactions():
    file_path = "/tmp/transactions.txt"
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    content = "Deposit,10000\nWithdraw,2500\nDeposit,4000\nWithdraw,1500\nDeposit,2000"
    with open(file_path, "w") as f:
        f.write(content.strip())
    print(f"✔ Created {file_path}")

def calculate_balance():
    file_path = "/tmp/transactions.txt"
    total_deposit = 0
    total_withdrawal = 0
    with open(file_path, "r") as f:
        for line in f:
            if line.strip():
                action, amount = line.strip().split(",")
                if action.strip() == "Deposit": total_deposit += int(amount)
                elif action.strip() == "Withdraw": total_withdrawal += int(amount)
    balance = total_deposit - total_withdrawal
    return total_deposit, total_withdrawal, balance

def generate_account_report():
    dep, withdr, bal = calculate_balance()
    report_path = "/tmp/account_report.txt"
    with open(report_path, "w") as f:
        f.write(f"Total Deposit = {dep}\nTotal Withdrawal = {withdr}\nFinal Balance = {bal}\n")
    print(f"✔ Generated Report at {report_path}")
    print("\n📋 File Content:")
    with open(report_path, "r") as f: print(f.read())

with DAG(dag_id='exercise_12_bank', start_date=datetime(2026, 1, 1), schedule=None, catchup=False) as dag:
    t1 = PythonOperator(task_id='create_transactions', python_callable=create_transactions)
    t2 = PythonOperator(task_id='calculate_balance', python_callable=calculate_balance)
    t3 = PythonOperator(task_id='generate_account_report', python_callable=generate_account_report)
    t1 >> t2 >> t3

# Trigger in Colab
create_transactions()
generate_account_report()

✔ Created /tmp/transactions.txt
✔ Generated Report at /tmp/account_report.txt

📋 File Content:
Total Deposit = 16000
Total Withdrawal = 4000
Final Balance = 12000

